# Sublimation curve from ASE thermochemistry

A gas phase from ASE can take a pressure. That makes it easy to go beyond the poster and repeat a transition calculation at many pressures.

We use a carbon-like element with made-up parameters: a harmonic crystal and a monatomic ideal gas. landau finds the sublimation temperature at each pressure, which gives the sublimation curve $p(T)$.

Saves `Sublimation_ASE.svg` and `.png`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from ase import Atoms
from ase.thermochemistry import HarmonicThermo, IdealGasThermo

import landau as ld
from landau.calculate import calc_phase_diagram
from landau.plot import plot_1d_T_phase_diagram

warnings.simplefilter("ignore")
P_ATM = 101325.0  # Pa

## Phases from ASE

In [ ]:
solid = ld.AsePhase(
    "solid",
    fixed_concentration=0.0,
    thermochem=HarmonicThermo(vib_energies=[0.15] * 3, potentialenergy=-7.4),
)

gas_thermo = IdealGasThermo(
    vib_energies=[], geometry="monatomic", atoms=Atoms("C"),
    symmetrynumber=1, spin=1, potentialenergy=0.0,
)


def gas(p):
    return ld.AsePhase("gas", fixed_concentration=0.0, thermochem=gas_thermo, pressure=p)

## Sublimation at 1 atm, then at every pressure

In [ ]:
Ts = np.linspace(1000, 8000, 60)
df_atm = calc_phase_diagram([solid, gas(P_ATM)], Ts=Ts, mu=0.0, keep_unstable=True)

pressures = np.logspace(0, 7, 15)  # Pa
T_sub = [
    calc_phase_diagram([solid, gas(p)], Ts=Ts, mu=0.0).query("border")["T"].iloc[0]
    for p in pressures
]
T_atm = df_atm.query("border")["T"].iloc[0]
print(f"Sublimation temperature at 1 atm: {T_atm:.0f} K")

In [ ]:
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(8, 3.4), layout="constrained")

plot_1d_T_phase_diagram(df_atm, ax=ax_a)
ax_a.set_xlabel("$T$ [K]")
ax_a.set_ylabel("$G$ [eV/atom]")
ax_a.set_title("a) free energies at 1 atm", loc="left")

ax_b.semilogy(T_sub, pressures, "o-", color="k", ms=4)
ax_b.fill_between(T_sub, pressures, pressures[-1], color="#ffb482", alpha=0.6, lw=0)
ax_b.fill_between(T_sub, pressures[0], pressures, color="#a1c9f4", alpha=0.6, lw=0)
ax_b.text(0.25, 0.75, "solid", transform=ax_b.transAxes, ha="center")
ax_b.text(0.75, 0.25, "gas", transform=ax_b.transAxes, ha="center")
ax_b.axhline(P_ATM, color="gray", ls="--", lw=1)
ax_b.plot(T_atm, P_ATM, "o", color="C3", zorder=3)
ax_b.set_xlim(min(T_sub), max(T_sub))
ax_b.set_ylim(pressures[0], pressures[-1])
ax_b.set_xlabel("$T$ [K]")
ax_b.set_ylabel("$p$ [Pa]")
ax_b.set_title("b) sublimation curve", loc="left")

fig.savefig("Sublimation_ASE.svg", transparent=True)
fig.savefig("Sublimation_ASE.png", transparent=True, dpi=300)